# DuaDeep Improved Pipeline - Cloud Execution
This notebook is configured to run the generalized DuaDeep model on a high-performance Linux cloud server (like NVIDIA DGX).
It assumes access to multi-GPU arrays or at least a single high-memory GPU.

In [ ]:
# Install Python Dependencies from the workspace
!pip install -r requirements.txt

In [ ]:
# Install MMseqs2 binaries for Linux environments (Required for structural sequence clustering)
import os

# Download and extract locally
!wget -nc https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz
!tar -xvfz mmseqs-linux-avx2.tar.gz

# Add local path to environment directly to bypass Colab specific hardcoding
mmseqs_path = os.path.abspath('mmseqs/bin')
os.environ['PATH'] = mmseqs_path + os.pathsep + os.environ.get('PATH', '')

# Test standard execution
!mmseqs --help

In [ ]:
# ----------------- ENCODERS -----------------
class AntibodyEncoder(nn.Module):
    """
    Antibody Pathway: Utilizes IgBERT to generate robust embeddings for Heavy/Light chains.
    """
    def __init__(self, model_name="Exscientia/IgBERT", freeze=True):
        super(AntibodyEncoder, self).__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        # Using add_pooling_layer=False to avoid missing 'pooler' weights in IgBERT
        self.encoder = AutoModel.from_pretrained(model_name, add_pooling_layer=False)
        
        if freeze:
            for param in self.encoder.parameters():
                param.requires_grad = False
                
    def forward(self, sequences):
        # Truncate explicitly to prevent CUDA OOM on uncharacteristically long sequence reads
        tokens = self.tokenizer(sequences, padding=True, truncation=True, max_length=512, return_tensors="pt")
        tokens = {k: v.to(next(self.parameters()).device) for k, v in tokens.items()}
        outputs = self.encoder(**tokens)
        
        # Typically use [CLS] token or mean pooling. Let's return mean pooled output.
        attention_mask = tokens['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        pooled = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return pooled

class AntigenEncoder(nn.Module):
    """
    Antigen Pathway: Utilizes ESM-Cambrian logic onto target antigens to glean 
    enriched biochemical features. Assumes Cambrian-style models accessible via HF.
    """
    def __init__(self, model_name="facebook/esm2_t33_650M_UR50D", freeze=True):
        super(AntigenEncoder, self).__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        
        if freeze:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward(self, sequences):
        # Prevent massive OOM spikes from ultra-long sequences
        tokens = self.tokenizer(sequences, padding=True, truncation=True, max_length=1024, return_tensors="pt")
        tokens = {k: v.to(next(self.parameters()).device) for k, v in tokens.items()}
        outputs = self.encoder(**tokens)
        
        attention_mask = tokens['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        pooled = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return pooled

# ----------------- NETWORK COMBINATION -----------------
class DuaDeepImprovedInteractionHead(nn.Module):
    def __init__(self, ab_dim=1024, ag_dim=1280, hidden_dims=[1024, 512, 128], num_classes=2):
        super(DuaDeepImprovedInteractionHead, self).__init__()
        fusion_dim = ab_dim + ag_dim
        layers = []
        in_dim = fusion_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            in_dim = h_dim
            
        self.mlp = nn.Sequential(*layers)
        self.classifier = nn.Linear(in_dim, num_classes)

    def forward(self, ab_embeds, ag_embeds):
        fused = torch.cat([ab_embeds, ag_embeds], dim=-1)
        out = self.mlp(fused)
        logits = self.classifier(out)
        return logits

class DuaDeepImproved(nn.Module):
    def __init__(self, ab_model="Exscientia/IgBERT", ag_model="facebook/esm2_t33_650M_UR50D", ab_freeze=True, ag_freeze=True):
        super(DuaDeepImproved, self).__init__()
        self.ab_encoder = AntibodyEncoder(model_name=ab_model, freeze=ab_freeze)
        self.ag_encoder = AntigenEncoder(model_name=ag_model, freeze=ag_freeze)
        
        # Output sizes: IgBERT is 1024 and ESM2-650M is 1280 
        self.interaction_head = DuaDeepImprovedInteractionHead(ab_dim=1024, ag_dim=1280)
        
    def forward(self, ab_seqs, ag_seqs):
        ab_emb = self.ab_encoder(ab_seqs)
        ag_emb = self.ag_encoder(ag_seqs)
        logits = self.interaction_head(ab_emb, ag_emb)
        return logits

In [ ]:
# ----------------- ENCODERS -----------------
class AntibodyEncoder(nn.Module):
    """
    Antibody Pathway: Utilizes IgBERT to generate robust embeddings for Heavy/Light chains.
    """
    def __init__(self, model_name="Exscientia/IgBERT", freeze=True):
        super(AntibodyEncoder, self).__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        # Using add_pooling_layer=False to avoid missing 'pooler' weights in IgBERT
        self.encoder = AutoModel.from_pretrained(model_name, add_pooling_layer=False)
        
        if freeze:
            for param in self.encoder.parameters():
                param.requires_grad = False
                
    def forward(self, sequences):
        # Truncate explicitly to prevent CUDA OOM on uncharacteristically long sequence reads
        tokens = self.tokenizer(sequences, padding=True, truncation=True, max_length=512, return_tensors="pt")
        tokens = {k: v.to(next(self.parameters()).device) for k, v in tokens.items()}
        outputs = self.encoder(**tokens)
        
        # Typically use [CLS] token or mean pooling. Let's return mean pooled output.
        attention_mask = tokens['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        pooled = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return pooled

class AntigenEncoder(nn.Module):
    """
    Antigen Pathway: Utilizes ESM-Cambrian logic onto target antigens to glean 
    enriched biochemical features. Assumes Cambrian-style models accessible via HF.
    """
    def __init__(self, model_name="facebook/esm2_t33_650M_UR50D", freeze=True):
        super(AntigenEncoder, self).__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        
        if freeze:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward(self, sequences):
        # Prevent massive OOM spikes from ultra-long sequences
        tokens = self.tokenizer(sequences, padding=True, truncation=True, max_length=1024, return_tensors="pt")
        tokens = {k: v.to(next(self.parameters()).device) for k, v in tokens.items()}
        outputs = self.encoder(**tokens)
        
        attention_mask = tokens['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        pooled = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return pooled

# ----------------- NETWORK COMBINATION -----------------
class DuaDeepImprovedInteractionHead(nn.Module):
    def __init__(self, ab_dim=768, ag_dim=1280, hidden_dims=[1024, 512, 128], num_classes=2):
        super(DuaDeepImprovedInteractionHead, self).__init__()
        fusion_dim = ab_dim + ag_dim
        layers = []
        in_dim = fusion_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            in_dim = h_dim
            
        self.mlp = nn.Sequential(*layers)
        self.classifier = nn.Linear(in_dim, num_classes)

    def forward(self, ab_embeds, ag_embeds):
        fused = torch.cat([ab_embeds, ag_embeds], dim=-1)
        out = self.mlp(fused)
        logits = self.classifier(out)
        return logits

class DuaDeepImproved(nn.Module):
    def __init__(self, ab_model="Exscientia/IgBERT", ag_model="facebook/esm2_t33_650M_UR50D", ab_freeze=True, ag_freeze=True):
        super(DuaDeepImproved, self).__init__()
        self.ab_encoder = AntibodyEncoder(model_name=ab_model, freeze=ab_freeze)
        self.ag_encoder = AntigenEncoder(model_name=ag_model, freeze=ag_freeze)
        
        # Output sizes: IgBERT is 768 and ESM2-650M is 1280 
        self.interaction_head = DuaDeepImprovedInteractionHead(ab_dim=768, ag_dim=1280)
        
    def forward(self, ab_seqs, ag_seqs):
        ab_emb = self.ab_encoder(ab_seqs)
        ag_emb = self.ag_encoder(ag_seqs)
        logits = self.interaction_head(ab_emb, ag_emb)
        return logits

In [ ]:
# ----------------- DATA PROCESSING -----------------
def run_mmseqs_clustering(fasta_path, out_dir, min_seq_id=0.8):
    os.makedirs(out_dir, exist_ok=True)
    db_path = os.path.join(out_dir, "db")
    cluster_path = os.path.join(out_dir, "clusters")
    
    # Run MMseqs locally ensuring explicit binary paths
    # Because we added mmseqs/bin to os.environ['PATH'], this subprocess should find it.
    subprocess.run(["mmseqs", "createdb", fasta_path, db_path], check=True)
    subprocess.run(["mmseqs", "cluster", db_path, cluster_path, out_dir, "--min-seq-id", str(min_seq_id)], check=True)
    
    tsv_out = os.path.join(out_dir, "clusters.tsv")
    subprocess.run(["mmseqs", "createtsv", db_path, db_path, cluster_path, tsv_out], check=True)
    return tsv_out

def setup_balanced_dataset(csv_path="AbRank_dataset.csv", undersample_threshold=1000):
    print(f"Loading {csv_path}...")
    df = pd.read_csv(csv_path, sep='\t')
    
    # Standardize schema mappings
    column_mapping = {
        'Ab_heavy_chain_seq': 'antibody',
        'Ag_seq': 'antigen',
        'Aff_op': 'label' 
    }
    actual_columns = {col: column_mapping[col] for col in column_mapping if col in df.columns}
    df = df.rename(columns=actual_columns)
    
    # Proxy formatting label logic for classification if dataset defines strength using '=' operations
    if 'label' in df.columns and df['label'].dtype == object:
         df['label'] = (df['label'] == '=').astype(int)
    
    # Formulate unique Ag sequences for memory clustering 
    unique_antigens = df['antigen'].unique()
    fasta_path = "antigens.fasta"
    with open(fasta_path, "w") as fw:
        for i, seq in enumerate(unique_antigens):
            fw.write(f">seq_{i}\n{seq}\n")
            
    print("Running MMseqs2 clustering...")
    try:
        tsv_path = run_mmseqs_clustering(fasta_path, "mmseqs_out")
        clusters = pd.read_csv(tsv_path, sep='\t', header=None, names=['cluster_rep', 'member'])
        cluster_map = dict(zip(clusters['member'], clusters['cluster_rep']))
        seq_to_cluster = {seq: cluster_map.get(f"seq_{i}", "unknown") for i, seq in enumerate(unique_antigens)}
    except FileNotFoundError:
        print("WARNING: MMseqs2 not found or PATH incomplete. Falling back to simple grouping.")
        seq_to_cluster = {seq: f"cluster_{i}" for i, seq in enumerate(unique_antigens)}
    
    df['cluster'] = df['antigen'].map(seq_to_cluster)
    cluster_counts = df['cluster'].value_counts()
    
    balanced_chunks = []
    # Identify heavily overrepresented clusters (HIV / SARS-CoV-2 strains)
    cluster_bar = tqdm(cluster_counts.items(), desc="Balancing Clusters")
    for c_id, count in cluster_bar:
        subset = df[df['cluster'] == c_id]
        if count > undersample_threshold:
            # Undersample the dense variant families
            subset = subset.sample(undersample_threshold, random_state=42)
            
        balanced_chunks.append(subset)
        
    df_balanced = pd.concat(balanced_chunks)
    print(f"Dataset Balanced to {len(df_balanced)} entries and prepared for zero-leakage training.")
    return df_balanced

# ----------------- EVALUATION & PLOTTING -----------------
def evaluate_and_plot(all_labels, all_preds, out_dir="results"):
    os.makedirs(out_dir, exist_ok=True)
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    binary_preds = (all_preds > 0.5).astype(int)

    print("Classification Report:")
    report = classification_report(all_labels, binary_preds, target_names=["Non-Binding", "Binding"])
    print(report)
    
    with open(os.path.join(out_dir, "classification_report.txt"), "w") as f:
        f.write(report)
        
    sns.set_theme(style="whitegrid")
    # ROC Curve 
    fpr, tpr, _ = roc_curve(all_labels, all_preds)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc="lower right")
    plt.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=300)
    plt.close()
    
    # PR Curve 
    precision, recall, _ = precision_recall_curve(all_labels, all_preds)
    pr_auc = average_precision_score(all_labels, all_preds)

    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, color='blue', lw=2, label=f'PR curve (AUPRC = {pr_auc:.4f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.legend(loc="lower left")
    plt.savefig(os.path.join(out_dir, "pr_curve.png"), dpi=300)
    plt.close()

    cm = confusion_matrix(all_labels, binary_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Non-Binding', 'Binding'], 
                yticklabels=['Non-Binding', 'Binding'])
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=300)
    plt.close()

    print(f"Evaluation plots saved to '{out_dir}/'")
    return roc_auc, pr_auc

In [ ]:
# ----------------- TRAINING LOOP -----------------
class BioInteractionDataset(Dataset):
    def __init__(self, ab_seqs, ag_seqs, labels):
        self.ab_seqs = ab_seqs
        self.ag_seqs = ag_seqs
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.ab_seqs[idx], self.ag_seqs[idx], self.labels[idx]

def train_duadeep(data_path="AbRank_dataset.csv", epochs=10, batch_size=32, lr=1e-4, weight_decay=1e-2):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Executing Training Pipeline on {device}")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        # Initialize Automatic Mixed Precision Scaler to halve memory requirements
        scaler = torch.amp.GradScaler('cuda')
    else:
        scaler = None
        
    # Dataset and Loader Configuration
    df = setup_balanced_dataset(data_path, undersample_threshold=5000)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    train_idx, val_idx = next(gss.split(df, groups=df['cluster']))
    
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    
    train_dataset = BioInteractionDataset(list(train_df['antibody']), list(train_df['antigen']), list(train_df['label']))
    val_dataset = BioInteractionDataset(list(val_df['antibody']), list(val_df['antigen']), list(val_df['label']))
    
    # We lowered the batch size to something reasonable like 16-32 here. If we need more we do gradient accumulation.
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    value_counts = train_df['label'].value_counts().sort_index().values
    samples_per_cls = value_counts.tolist()
    
    # Initialize Core Topologies mapped with explicit freezing layers.
    model = DuaDeepImproved(ab_freeze=True, ag_freeze=True)
    model.to(device)
    
    classes_list = torch.tensor(samples_per_cls).float().to(device)
    criterion = ClassBalancedFocalLoss(samples_per_cls=classes_list, num_classes=2, beta=0.9999)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    os.makedirs('checkpoints', exist_ok=True)
    best_val_auc = 0.0
    
    # Step 4: Mixed Precision Epoch Traversals
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        train_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}] [Train]")
        
        for ab_batch, ag_batch, labels_batch in train_bar:
            labels_batch = labels_batch.to(device)
            optimizer.zero_grad()
            
            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    logits = model(ab_batch, ag_batch)
                    loss = criterion(labels_batch, logits)
                    
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(ab_batch, ag_batch)
                loss = criterion(labels_batch, logits)
                loss.backward()
                optimizer.step()
                
            total_loss += loss.item()
            train_bar.set_postfix(loss=loss.item())
            
            # Explicit cleanup of intermediate tensors that pin GPU memory
            del logits, loss, labels_batch, ab_batch, ag_batch
        
        print(f"Epoch [{epoch+1}/{epochs}] Final Loss: {total_loss/len(train_loader):.4f}")
        
        # Validation Routine
        model.eval()
        val_loss, all_labels, all_preds = 0, [], []
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{epochs}] [Val]")
            for ab_batch, ag_batch, labels_batch in val_bar:
                # Force everything that's mapped to cuda immediately 
                labels_batch = labels_batch.to(device)
                
                if scaler is not None:
                     with torch.amp.autocast('cuda'):
                         logits = model(ab_batch, ag_batch)
                         loss = criterion(labels_batch, logits)
                else:
                     logits = model(ab_batch, ag_batch)
                     loss = criterion(labels_batch, logits)
                     
                val_loss += loss.item()
                val_bar.set_postfix(loss=loss.item())
                
                probs = torch.softmax(logits, dim=1)[:, 1]
                all_labels.extend(labels_batch.cpu().numpy())
                all_preds.extend(probs.cpu().numpy())
                
                del logits, loss, labels_batch, ab_batch, ag_batch

        val_roc = roc_auc_score(all_labels, all_preds)
        val_prauc = average_precision_score(all_labels, all_preds)
        val_preds = (np.array(all_preds) > 0.5).astype(int)
        val_f1 = f1_score(all_labels, val_preds)
        
        print(f"Val ROC-AUC: {val_roc:.4f} | Val AUPRC: {val_prauc:.4f} | Val F1: {val_f1:.4f}\n")
        
        if val_roc > best_val_auc:
            best_val_auc = val_roc
            chkpt_path = os.path.join('checkpoints', 'best_model.pth')
            torch.save(model.state_dict(), chkpt_path)
            print(f"--> Saved better model with ROC-AUC {val_roc:.4f} @ epoch {epoch+1}")
            
        torch.cuda.empty_cache()

    # Step 5: Evaluation Reports Mapped Across Extrapolated Boundaries 
    print("Training Completed. Formatting evaluation reports on Best Generalization Module...")
    best_model = DuaDeepImproved(ab_freeze=True, ag_freeze=True)
    best_model.load_state_dict(torch.load(os.path.join('checkpoints', 'best_model.pth')))
    best_model.to(device)
    best_model.eval()
    
    final_labels, final_preds = [], []
    with torch.no_grad():
        for ab_batch, ag_batch, labels_batch in val_loader:
            labels_batch = labels_batch.to(device)
            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    logits = best_model(ab_batch, ag_batch)
            else:
                logits = best_model(ab_batch, ag_batch)
                
            probs = torch.softmax(logits, dim=1)[:, 1]
            final_labels.extend(labels_batch.cpu().numpy())
            final_preds.extend(probs.cpu().numpy())
            
    evaluate_and_plot(final_labels, final_preds, out_dir="results")

In [ ]:
# Execute the Training Pipeline with memory-safe dimensions mapping properly across the GPU context
print("Starting DuaDeep Improved run...")
train_duadeep(
    data_path="AbRank_dataset.csv",
    epochs=10,
    batch_size=16,      # Safely lowered to 16 to prevent CUDA OOM exceptions (IgBERT + ESM-2 combined is heavily taxing).
    lr=1e-4, 
    weight_decay=1e-2
)